# Interactive RAG Preparation Lab
## Cleaning, Chunking, Retrieval, and Prompt Augmentation

This notebook visualizes how raw document text becomes retrieval-ready RAG data.

You will compare cleaning stages, fixed chunks, overlap, sentence chunks, recursive splitting, retrieval results, and the final grounded prompt.

# RAG Pipeline

```text
Documents → Load → Clean → Chunk → Embed → Store
          → Retrieve → Augment → Generate
```

In [ ]:
stages = ["Load","Clean","Chunk","Embed","Store","Retrieve","Augment","Generate"]
for i, stage in enumerate(stages, 1):
    print(i, stage)

# Messy Sample Document

In [ ]:
raw_text = """EMPLOYEE HANDBOOK
1

Vacation Policy

Full-time employees receive 15 paid
vacation days every calendar year.     Employees must submit
requests to their manager.

EMPLOYEE HANDBOOK
2

Remote Work Policy

Employees may work remotely up to two days
per week.	Manager approval is required.

EMPLOYEE HANDBOOK
3

Security Policy

Passwords must not be shared. Confidential files
must be stored in approved systems.
"""
print(raw_text)

# Cleaning Functions

In [ ]:
import re

def remove_control_characters(text):
    return text.replace("\\x00", "").replace("\\u00a0", " ")

def normalize_spaces(text):
    return re.sub(r"[ \t]+", " ", text)

def remove_page_number_lines(text):
    return "\n".join(line for line in text.splitlines() if not line.strip().isdigit())

def remove_known_headers(text, headers):
    for header in headers:
        text = text.replace(header, "")
    return text

def repair_line_breaks(text):
    return re.sub(r"(?<!\n)\n(?=[a-z])", " ", text)

def normalize_blank_lines(text):
    return re.sub(r"\n{3,}", "\n\n", text)

def clean_text(text):
    text = remove_control_characters(text)
    text = remove_known_headers(text, ["EMPLOYEE HANDBOOK"])
    text = remove_page_number_lines(text)
    text = normalize_spaces(text)
    text = repair_line_breaks(text)
    text = normalize_blank_lines(text)
    return text.strip()

# View Every Cleaning Stage

In [ ]:
values = []
x = remove_control_characters(raw_text); values.append(("Remove control characters", x))
x = remove_known_headers(x, ["EMPLOYEE HANDBOOK"]); values.append(("Remove header", x))
x = remove_page_number_lines(x); values.append(("Remove page numbers", x))
x = normalize_spaces(x); values.append(("Normalize spaces", x))
x = repair_line_breaks(x); values.append(("Repair line breaks", x))
x = normalize_blank_lines(x).strip(); values.append(("Normalize blank lines", x))

for name, text in values:
    print("\n" + "="*70)
    print(name)
    print(text)

In [ ]:
cleaned_text = clean_text(raw_text)
print("Raw characters:", len(raw_text))
print("Cleaned characters:", len(cleaned_text))
print(cleaned_text)

# Fixed-Size Chunking

In [ ]:
def fixed_chunks(text, size=150):
    return [text[i:i+size] for i in range(0, len(text), size)]

for size in [20, 150, 500]:
    chunks = fixed_chunks(cleaned_text, size)
    print(f"\nSize {size}: {len(chunks)} chunks")
    for i, chunk in enumerate(chunks[:4], 1):
        print(i, repr(chunk))

# Sliding-Window Chunking

In [ ]:
def overlapping_chunks(text, size=180, overlap=40):
    if overlap < 0 or overlap >= size:
        raise ValueError("Overlap must be smaller than chunk size.")
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start:start+size])
        if start + size >= len(text):
            break
        start += size - overlap
    return chunks

for overlap in [0, 30, 80]:
    chunks = overlapping_chunks(cleaned_text, 180, overlap)
    print("Overlap:", overlap, "| chunks:", len(chunks))

# Sentence-Based Chunking

In [ ]:
def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def sentence_chunks(text, count=2):
    sentences = split_sentences(text)
    return [" ".join(sentences[i:i+count]) for i in range(0, len(sentences), count)]

for i, chunk in enumerate(sentence_chunks(cleaned_text, 2), 1):
    print(f"\nChunk {i}:")
    print(chunk)

# Recursive-Style Chunking

In [ ]:
def recursive_split(text, max_size=220, separators=None):
    if separators is None:
        separators = ["\n\n", ". ", " ", ""]
    text = text.strip()
    if not text:
        return []
    if len(text) <= max_size:
        return [text]
    separator = separators[0]
    if separator == "":
        return fixed_chunks(text, max_size)

    parts, chunks, current = text.split(separator), [], ""
    for part in parts:
        candidate = part if not current else current + separator + part
        if len(candidate) <= max_size:
            current = candidate
        else:
            if current:
                chunks.extend(recursive_split(current, max_size, separators[1:]))
            current = part
    if current:
        chunks.extend(recursive_split(current, max_size, separators[1:]))
    return [chunk.strip() for chunk in chunks if chunk.strip()]

recursive_chunks = recursive_split(cleaned_text, 220)
for i, chunk in enumerate(recursive_chunks, 1):
    print(f"\nChunk {i} ({len(chunk)} characters)")
    print(chunk)

# Compare Strategies

In [ ]:
import pandas as pd

strategies = {
    "Fixed 180": fixed_chunks(cleaned_text, 180),
    "Overlap 180/50": overlapping_chunks(cleaned_text, 180, 50),
    "Two sentences": sentence_chunks(cleaned_text, 2),
    "Recursive 220": recursive_split(cleaned_text, 220)
}

rows = []
for name, chunks in strategies.items():
    lengths = [len(c) for c in chunks]
    rows.append({
        "strategy": name,
        "chunks": len(chunks),
        "minimum": min(lengths),
        "maximum": max(lengths),
        "average": round(sum(lengths)/len(lengths), 1)
    })

display(pd.DataFrame(rows))

# Chunk Metadata

In [ ]:
chunk_records = [{
    "chunk_id": f"handbook_{i:03d}",
    "document": "employee_handbook",
    "chunk_index": i,
    "character_count": len(chunk),
    "text": chunk
} for i, chunk in enumerate(recursive_chunks)]

display(pd.DataFrame(chunk_records))

# Local Retrieval Simulation

In [ ]:
import math
from collections import Counter

STOPWORDS = {"the","a","an","and","or","to","of","in","is","are","be","for","on","with","this","that","how","many","do","does","what"}

def tokenize(text):
    return [w for w in re.findall(r"[a-zA-Z0-9]+", text.lower()) if w not in STOPWORDS]

def retrieval_score(query, chunk):
    q, c = Counter(tokenize(query)), Counter(tokenize(chunk))
    shared = set(q) & set(c)
    if not shared:
        return 0.0
    return sum(min(q[w], c[w]) for w in shared) / math.sqrt(max(1, len(tokenize(chunk))))

def retrieve(query, records, top_k=3):
    scored = [{**r, "score": retrieval_score(query, r["text"])} for r in records]
    return sorted(scored, key=lambda r: r["score"], reverse=True)[:top_k]

In [ ]:
query = "How many vacation days do full-time employees receive?"
retrieved = retrieve(query, chunk_records, 3)

for rank, record in enumerate(retrieved, 1):
    print(f"Rank {rank} | Score {record['score']:.3f}")
    print(record["text"], "\n")

# Build the Augmented Prompt

In [ ]:
def build_prompt(query, records):
    context = "\n\n".join(
        f"[Chunk {r['chunk_index']}]\n{r['text']}"
        for r in records
    )
    return (
        "You are a document-grounded assistant.\n\n"
        "Use only the context below.\n"
        "If the answer is unsupported, say:\n"
        '"I do not know based on the uploaded documents."\n\n'
        f"CONTEXT:\n{context}\n\n"
        f"QUESTION:\n{query}\n\nANSWER:"
    )

print(build_prompt(query, retrieved))

# Test Grounding Behavior

In [ ]:
tests = [
    "How many vacation days do employees receive?",
    "Does the company provide dental insurance?"
]

for question in tests:
    results = retrieve(question, chunk_records, 3)
    supported = any(r["score"] >= 0.15 for r in results)
    print("\nQuestion:", question)
    print("Context may support an answer." if supported
          else "I do not know based on the uploaded documents.")

# Reflection

- Which cleaning step made the biggest difference?
- Which chunking strategy preserved meaning best?
- What did overlap change?
- Why should chunks retain metadata?
- What should happen when retrieval is weak?